# Step 6 — Full-scale prediction and evaluation prep

Runs **locally** (the six trained models are small and every input is on local disk — no cluster, no sbatch). It turns the six full-scale Skip-gram models into per-config judging tasks — one plain markdown table per configuration for an LLM to judge by hand.

Pipeline:
1. **Anchor selection** — 12 department-diverse anchors from the six models' shared vocabulary.
2. **Top-10 per model per anchor** — each model's own recommendations.
3. **Per-config candidate sets** — each config's own top-10 per anchor, candidates labelled `C01`–`C10` in rank order.
4. **Co-occurrence** — from `order_products__prior.csv`, for the novelty metric (computed from data for the local key, not shown to the judge).
5. **Write six per-config files + one key** — `pairs_pct_dropoff_0.03.md … pairs_kneedle_1.0.md` (one flat table each) to paste into a chat, plus `evaluation_key.json` kept as a local record.

**Author-only — do not execute end-to-end until the six full-scale models exist** at `outputs/full_scale/skipgram/{method}/{value_folder}/` (produced by `run_skipgram_fullscale_cluster.py`).

## 1. Imports, paths, and configuration

Paths resolve relative to the project root so the notebook works regardless of where Jupyter is launched from. `CONFIGS` is replicated verbatim from `run_skipgram_fullscale_cluster.py` (same structure, same `value_folder` names) — not invented here.

**Note on `products_text.csv`.** The task points at `outputs/products_text.csv`, but the earlier outputs reorganisation moved it to `outputs/step1/products_text.csv`. `load_products_text()` below tries both locations, and if neither carries the needed columns (name / aisle / department) falls back to joining `data/products.csv` + `data/aisles.csv` + `data/departments.csv`.

In [1]:
import json
import datetime
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs"
FULL_SCALE_DIR = OUT_DIR / "full_scale"
SKIPGRAM_DIR = FULL_SCALE_DIR / "skipgram"
EVAL_DIR = FULL_SCALE_DIR / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# The six configurations — replicated verbatim from run_skipgram_fullscale_cluster.py
# (same structure + value_folder names; matches the on-disk folders).
CONFIGS = [
    {"method": "pct_dropoff", "value": 0.03, "param": "drop_pct",
     "value_folder": "drop_0.03", "label": "pct_dropoff (0.03)"},
    {"method": "pct_dropoff", "value": 0.05, "param": "drop_pct",
     "value_folder": "drop_0.05", "label": "pct_dropoff (0.05)"},
    {"method": "pct_dropoff", "value": 0.10, "param": "drop_pct",
     "value_folder": "drop_0.10", "label": "pct_dropoff (0.10)"},
    {"method": "kneedle", "value": 0.3, "param": "knee_sensitivity",
     "value_folder": "sensitivity_0.3", "label": "kneedle (0.3)"},
    {"method": "kneedle", "value": 0.5, "param": "knee_sensitivity",
     "value_folder": "sensitivity_0.5", "label": "kneedle (0.5)"},
    {"method": "kneedle", "value": 1.0, "param": "knee_sensitivity",
     "value_folder": "sensitivity_1.0", "label": "kneedle (1.0)"},
]
CONFIG_LABELS = [c["label"] for c in CONFIGS]

# ---- Selection / evaluation parameters ----
SELECTION_SEED = 11        # np.random.default_rng seed for the per-department pick
MIN_BASKET_COUNT = 50      # exclude near-orphan anchors from this first pass
TOP_K = 10                 # top-K recommendations per model per anchor
NOVEL_MAX_COOCC = 0        # "novel" = judged substitute with co-occurrence <= this (zero)
FORCE_RESELECT = False     # if True, recompute anchors even if anchors.json exists

# One anchor from each of these departments (matched against the real strings below).
TARGET_DEPARTMENTS = [
    "dairy eggs", "produce", "frozen", "bakery", "meat seafood", "pantry",
    "snacks", "beverages", "personal care", "household", "alcohol", "canned goods",
]

print("Outputs dir     :", OUT_DIR)
print("Full-scale dir  :", FULL_SCALE_DIR)
print("Evaluation dir  :", EVAL_DIR)

Outputs dir     : /Users/linnhtet/Developer/bachelor-thesis/outputs
Full-scale dir  : /Users/linnhtet/Developer/bachelor-thesis/outputs/full_scale
Evaluation dir  : /Users/linnhtet/Developer/bachelor-thesis/outputs/full_scale/evaluation


In [2]:
def load_products_text():
    """Product metadata (name, aisle, department). Tries outputs/products_text.csv,
    then outputs/step1/products_text.csv (the reorg location), then falls back to
    joining data/products.csv + aisles.csv + departments.csv if columns are missing."""
    needed = {"product_id", "product_name", "aisle", "department"}
    for p in (OUT_DIR / "products_text.csv", OUT_DIR / "step1" / "products_text.csv"):
        if p.exists():
            df = pd.read_csv(p)
            if needed <= set(df.columns):
                print(f"products_text: using {p}")
                return df
            print(f"products_text at {p} is missing {needed - set(df.columns)}; trying next source")
    print("products_text: falling back to data/products.csv + aisles.csv + departments.csv join")
    prod = pd.read_csv(DATA_DIR / "products.csv")
    ais = pd.read_csv(DATA_DIR / "aisles.csv")
    dep = pd.read_csv(DATA_DIR / "departments.csv")
    return prod.merge(ais, on="aisle_id", how="left").merge(dep, on="department_id", how="left")


products_text = load_products_text()
id_to_name = dict(zip(products_text["product_id"], products_text["product_name"]))
id_to_aisle = dict(zip(products_text["product_id"], products_text["aisle"]))
id_to_dept = dict(zip(products_text["product_id"], products_text["department"]))


def name(pid):  return id_to_name.get(int(pid), f"?{pid}")
def aisle(pid): return id_to_aisle.get(int(pid), "?")
def dept(pid):  return id_to_dept.get(int(pid), "?")


# Per-product basket counts (str keys in the JSON -> int).
with open(FULL_SCALE_DIR / "product_basket_freq.json") as f:
    _freq = json.load(f)
basket_count = {int(pid): c for pid, c in _freq["product_basket_count"].items()}
print(f"basket counts loaded: {len(basket_count):,} products, "
      f"{_freq['total_baskets']:,} total baskets")


def load_vocab(cfg):
    """The set of product_ids that have a trained vector under one config."""
    with open(SKIPGRAM_DIR / cfg["method"] / cfg["value_folder"] / "product_id_to_index.json") as f:
        return {int(pid) for pid in json.load(f).keys()}


vocabs = {c["label"]: load_vocab(c) for c in CONFIGS}
intersection = set.intersection(*vocabs.values())
print("\nVocabulary size per config:")
for c in CONFIGS:
    print(f"  {c['label']:<22} {len(vocabs[c['label']]):,}")
_smallest = min(CONFIGS, key=lambda c: len(vocabs[c["label"]]))
print(f"\nsmallest vocab (bounds the intersection): {_smallest['label']} "
      f"({len(vocabs[_smallest['label']]):,})")
print(f"anchors comparable across ALL SIX (intersection): {len(intersection):,}")

products_text: using /Users/linnhtet/Developer/bachelor-thesis/outputs/step1/products_text.csv
basket counts loaded: 49,677 products, 3,214,874 total baskets

Vocabulary size per config:
  pct_dropoff (0.03)     32,295
  pct_dropoff (0.05)     39,104
  pct_dropoff (0.10)     43,851
  kneedle (0.3)          44,765
  kneedle (0.5)          45,372
  kneedle (1.0)          48,887

smallest vocab (bounds the intersection): pct_dropoff (0.03) (32,295)
anchors comparable across ALL SIX (intersection): 31,585


## Step 1 — Anchor selection

The **intersection of all six models' vocabularies**: only anchors present in every model can be compared fairly, and `pct_dropoff (0.03)` — with the smallest vocabulary — bounds that intersection.

From the intersection we take **12 anchors, one per department** (the twelve listed above). Within each department we restrict to products with `basket_count >= 50` and pick one with `np.random.default_rng(11)`. Each requested department is matched against the real department strings; any that fail to match are printed, not silently skipped. The selection is saved to `outputs/full_scale/evaluation/anchors.json` so it is reproducible and reusable (and reloaded as-is on later runs unless `FORCE_RESELECT = True`).

> **Thin anchors are deliberately excluded here.** Products with `basket_count < 50` are left out of this first pass to avoid degenerate near-orphan anchors. They are exactly where a substitute-finding method is most likely to struggle, so they deserve their **own separate evaluation pass** later rather than being diluted into this department-diverse comparison.

In [3]:
def select_anchors():
    """Pick one intersection anchor per target department (basket_count >= 50),
    deterministically via np.random.default_rng(SELECTION_SEED). Returns the anchor
    list (dicts) plus the resolved/unmatched department bookkeeping."""
    actual_depts = set(str(d) for d in products_text["department"].dropna().unique())

    # Resolve each requested department string against the real ones (exact first,
    # then a case/space-insensitive attempt); report anything that fails to match.
    resolved = []          # (requested, actual)
    unmatched = []
    norm = {d.strip().lower(): d for d in actual_depts}
    for want in TARGET_DEPARTMENTS:
        if want in actual_depts:
            resolved.append((want, want))
        elif want.strip().lower() in norm:
            resolved.append((want, norm[want.strip().lower()]))
        else:
            unmatched.append(want)
    if unmatched:
        print("!! departments with NO match (skipped):", unmatched)

    rng = np.random.default_rng(SELECTION_SEED)
    anchors = []
    for i, (want, actual) in enumerate(resolved, start=1):
        pool = sorted(pid for pid in intersection
                      if id_to_dept.get(pid) == actual and basket_count.get(pid, 0) >= MIN_BASKET_COUNT)
        if not pool:
            print(f"!! no eligible anchor for department '{actual}' "
                  f"(intersection ∩ basket_count>={MIN_BASKET_COUNT} is empty) — skipped")
            continue
        pid = int(rng.choice(np.array(pool)))
        anchors.append({
            "label": f"A{len(anchors) + 1:02d}",
            "product_id": pid,
            "name": name(pid),
            "aisle": aisle(pid),
            "department": actual,
            "basket_count": int(basket_count.get(pid, 0)),
        })
    return anchors, resolved, unmatched


anchors_path = EVAL_DIR / "anchors.json"
if anchors_path.exists() and not FORCE_RESELECT:
    with open(anchors_path) as f:
        anchors = json.load(f)["anchors"]
    print(f"Reusing existing selection: {anchors_path} ({len(anchors)} anchors). "
          f"Set FORCE_RESELECT=True to redraw.\n")
else:
    anchors, _resolved, _unmatched = select_anchors()
    payload = {
        "selection_seed": SELECTION_SEED,
        "min_basket_count": MIN_BASKET_COUNT,
        "intersection_size": len(intersection),
        "bounding_config": _smallest["label"],
        "target_departments": TARGET_DEPARTMENTS,
        "unmatched_departments": _unmatched,
        "n_anchors": len(anchors),
        "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
        "anchors": anchors,
    }
    with open(anchors_path, "w") as f:
        json.dump(payload, f, indent=2)
    print(f"Saved {anchors_path} ({len(anchors)} anchors)\n")

print(f"{'label':<6}{'id':>8}  {'dept':<16}{'aisle':<26}{'baskets':>9}  name")
print("-" * 100)
for a in anchors:
    print(f"{a['label']:<6}{a['product_id']:>8}  {a['department']:<16}{a['aisle'][:24]:<26}"
          f"{a['basket_count']:>9,}  {a['name']}")

anchor_ids = [a["product_id"] for a in anchors]

Saved /Users/linnhtet/Developer/bachelor-thesis/outputs/full_scale/evaluation/anchors.json (12 anchors)

label       id  dept            aisle                       baskets  name
----------------------------------------------------------------------------------------------------
A01       6719  dairy eggs      packaged cheese               1,055  Mini Babybel White Cheddar Cheese
A02       5895  produce         fresh vegetables                137  White Asparagus
A03      39664  frozen          frozen appetizers sides         268  Crispy Chicken Strips
A04      24731  bakery          tortillas flat bread            104  Flour Burrito Caseras Tortillas
A05      28900  meat seafood    hot dogs bacon sausage          319  Black Label Center Cut Bacon
A06      30287  pantry          salad dressing toppings         990  Ranch Original Topping & Dressing
A07      35942  snacks          energy granola bars             672  Mojo Peanut Butter Bar
A08       1576  beverages       coffee         

## Step 2 — Top-10 per model per anchor

For each model: load its numpy pair (`embeddings.npy` + `product_id_to_index.json` — the input vectors only, not `model.gensim`), unit-normalise the matrix once so cosine is a plain dot product, and for each anchor compute similarity against every other product in that model's vocabulary. The anchor's own row is masked to `-inf` **before** sorting (same discipline as `filter_lib_fullscale.anchor_sorted_sims`), then we take the top 10.

In [4]:
def load_model_vectors(cfg):
    """Unit-normalised embedding matrix + id maps for one config (numpy pair only)."""
    d = SKIPGRAM_DIR / cfg["method"] / cfg["value_folder"]
    emb = np.load(d / "embeddings.npy").astype(np.float32)
    emb = emb / np.clip(np.linalg.norm(emb, axis=1, keepdims=True), 1e-12, None)
    with open(d / "product_id_to_index.json") as f:
        pid_to_idx = {int(pid): i for pid, i in json.load(f).items()}
    idx_to_pid = {i: pid for pid, i in pid_to_idx.items()}
    return emb, pid_to_idx, idx_to_pid


# top10[config_label][anchor_pid] = [(candidate_pid, score), ...] length TOP_K
top10 = {c["label"]: {} for c in CONFIGS}

for c in CONFIGS:
    emb, pid_to_idx, idx_to_pid = load_model_vectors(c)
    for apid in anchor_ids:
        a_idx = pid_to_idx[apid]                      # anchor is in the intersection -> present
        sims = emb @ emb[a_idx]                       # cosine (unit-norm) vs every product
        sims[a_idx] = -np.inf                         # mask self BEFORE sorting
        order = np.argsort(-sims)[:TOP_K]             # top-K indices, descending
        top10[c["label"]][apid] = [(int(idx_to_pid[int(i)]), float(sims[int(i)])) for i in order]
    print(f"  {c['label']:<22} top-{TOP_K} computed for {len(anchor_ids)} anchors")

print("\nStep 2 done.")

  pct_dropoff (0.03)     top-10 computed for 12 anchors
  pct_dropoff (0.05)     top-10 computed for 12 anchors
  pct_dropoff (0.10)     top-10 computed for 12 anchors
  kneedle (0.3)          top-10 computed for 12 anchors
  kneedle (0.5)          top-10 computed for 12 anchors
  kneedle (1.0)          top-10 computed for 12 anchors

Step 2 done.


## Step 3 — Per-config candidate sets

The union/dedupe is dropped: it made per-config comparison impossible, which is the whole point of the evaluation. Instead **each config keeps its own ordered top-10 per anchor**, exactly as Step 2 produced it.

Within each `(config, anchor)`, candidates are labelled `C01..C10` in the model's own ranked order (rank 1 first). Labels are scoped per anchor per config, so `C01` under one config and `C01` under another are different products — expected.

In [5]:
# per_config[config_label][anchor_label] = [ {clabel, product_id, rank, score}, ... ]
# in the model's own ranked order (rank 1 first).
per_config = {c["label"]: {} for c in CONFIGS}
for c in CONFIGS:
    label = c["label"]
    for a in anchors:
        apid = a["product_id"]
        per_config[label][a["label"]] = [
            {"clabel": f"C{rank:02d}", "product_id": int(cpid), "rank": rank, "score": float(score)}
            for rank, (cpid, score) in enumerate(top10[label][apid], start=1)
        ]

n_rows = sum(len(per_config[c["label"]][a["label"]]) for c in CONFIGS for a in anchors)
print(f"{len(CONFIGS)} configs x {len(anchors)} anchors x up to {TOP_K} candidates "
      f"= {n_rows:,} (config, anchor, candidate) rows to judge")

6 configs x 12 anchors x up to 10 candidates = 720 (config, anchor, candidate) rows to judge


## Step 4 — Co-occurrence over the per-config candidates

For the novelty metric: how many baskets contain both the anchor and a candidate. The pair set is the **union of all six configs'** `(anchor, candidate)` pairs — the same product often appears under several configs, so each unique pair is counted once here and looked up per config when writing. As before, `order_products__prior.csv` is filtered to just the anchors plus every candidate product **while reading**, never building the full graph.

In [6]:
# Union of every config's (anchor, candidate) pairs; a product surfaced by several
# configs is counted once, then looked up per config at write time.
relevant_pids = set(anchor_ids)
anchor_candidate_pids = defaultdict(set)     # anchor_pid -> set(candidate_pid) across all configs
for c in CONFIGS:
    for a in anchors:
        for entry in per_config[c["label"]][a["label"]]:
            anchor_candidate_pids[a["product_id"]].add(entry["product_id"])
            relevant_pids.add(entry["product_id"])
print(f"relevant products for co-occurrence: {len(relevant_pids):,} "
      f"({len(anchor_ids)} anchors + all configs' candidate products)")

# product_id -> set(order_id), streaming the prior and keeping only relevant rows.
pid_orders = defaultdict(set)
n_rows = 0
prior_path = DATA_DIR / "order_products__prior.csv"
for chunk in pd.read_csv(prior_path, usecols=["order_id", "product_id"],
                         dtype={"order_id": "int32", "product_id": "int32"},
                         chunksize=2_000_000):
    n_rows += len(chunk)
    hit = chunk[chunk["product_id"].isin(relevant_pids)]
    for oid, pid in zip(hit["order_id"].to_numpy(), hit["product_id"].to_numpy()):
        pid_orders[int(pid)].add(int(oid))
    print(f"  scanned {n_rows:,} rows...", end="\r")
print(f"\nscanned {n_rows:,} rows; have order-sets for {len(pid_orders):,} relevant products")

# co_occ[(anchor_pid, candidate_pid)] = baskets containing BOTH (each unique pair once).
co_occ = {}
for apid, cset in anchor_candidate_pids.items():
    a_orders = pid_orders.get(apid, set())
    for cpid in cset:
        co_occ[(apid, cpid)] = len(a_orders & pid_orders.get(cpid, set()))
_zeros = sum(1 for v in co_occ.values() if v == 0)
print(f"co-occurrence computed for {len(co_occ):,} unique (anchor, candidate) pairs "
      f"({_zeros:,} have zero co-occurrence)")

relevant products for co-occurrence: 571 (12 anchors + all configs' candidate products)
  scanned 32,434,489 rows...
scanned 32,434,489 rows; have order-sets for 571 relevant products
co-occurrence computed for 561 unique (anchor, candidate) pairs (489 have zero co-occurrence)


## Step 5 — Write six per-config files plus one local key

Six markdown files, one per config, **named by the real config**: `pairs_pct_dropoff_0.03.md` … `pairs_kneedle_1.0.md`.

Each file is **one flat markdown table and nothing else** — no heading, no judging instructions, no per-anchor subheadings, no preamble. Columns: `Anchor | Candidate | Candidate Aisle | Candidate Department`. The anchor name repeats across its ten candidate rows; candidates stay in the model's own ranked order (rank 1 first); the aisle/department shown are the **candidate's** (the judging rubric needs that context). No co-occurrence, product_ids, similarity scores, or rank numbers appear in these files.

`evaluation_key.json` is kept as the **local record** (not given to the judge), keyed by real config name, holding per anchor/candidate the product_id, rank, similarity, and co-occurrence count — the last still needed for the novelty rate, now computed from data rather than shown to the judge.

In [7]:
def cooc(apid, cpid):
    return co_occ[(apid, cpid)]


def config_filename(cfg):
    """Real-config file name: pct_dropoff (0.03) -> pairs_pct_dropoff_0.03.md,
    kneedle (0.3) -> pairs_kneedle_0.3.md. The numeric suffix is taken from the
    config's value_folder so it matches the on-disk 0.03 / 0.3 / 1.0 formatting."""
    value_str = cfg["value_folder"].split("_", 1)[1]     # "drop_0.03" -> "0.03"
    return f"pairs_{cfg['method']}_{value_str}.md"


# ---- (a) six per-config markdown files: ONE flat table each, named by real config ----
for cfg in CONFIGS:
    path = EVAL_DIR / config_filename(cfg)
    with open(path, "w") as f:
        f.write("| Anchor | Candidate | Candidate Aisle | Candidate Department |\n")
        f.write("|---|---|---|---|\n")
        for a in anchors:
            for entry in per_config[cfg["label"]][a["label"]]:   # model's own ranked order (rank 1 first)
                cpid = entry["product_id"]
                f.write(f"| {a['name']} | {name(cpid)} | {aisle(cpid)} | {dept(cpid)} |\n")
    print(f"Saved {path.name}")

# ---- (b) local record evaluation_key.json, keyed by REAL config name (withheld from judge) ----
key = {
    "meta": {
        "selection_seed": SELECTION_SEED,
        "min_basket_count": MIN_BASKET_COUNT,
        "top_k": TOP_K,
        "novel_max_cooccurrence": NOVEL_MAX_COOCC,
        "config_labels": CONFIG_LABELS,
        "note": "Novelty = judge says substitute AND co_occurrence <= novel_max_cooccurrence.",
        "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
    },
    "configs": {},
}
for cfg in CONFIGS:
    cfg_entry = {"anchors": {}}
    for a in anchors:
        apid = a["product_id"]
        cand = {}
        for entry in per_config[cfg["label"]][a["label"]]:
            cpid = entry["product_id"]
            cand[entry["clabel"]] = {
                "product_id": cpid,
                "name": name(cpid),
                "rank": entry["rank"],
                "score": entry["score"],
                "co_occurrence": cooc(apid, cpid),
            }
        cfg_entry["anchors"][a["label"]] = {
            "anchor_product_id": apid,
            "anchor_name": a["name"],
            "candidates": cand,
        }
    key["configs"][cfg["label"]] = cfg_entry

key_path = EVAL_DIR / "evaluation_key.json"
with open(key_path, "w") as f:
    json.dump(key, f, indent=2)
print(f"\nSaved {key_path.name} (local record, keyed by real config name; not given to the judge)")

Saved pairs_pct_dropoff_0.03.md
Saved pairs_pct_dropoff_0.05.md
Saved pairs_pct_dropoff_0.10.md
Saved pairs_kneedle_0.3.md
Saved pairs_kneedle_0.5.md
Saved pairs_kneedle_1.0.md

Saved evaluation_key.json (local record, keyed by real config name; not given to the judge)
